# Evaluation metrics (offline)

This notebook computes **accuracy, precision, recall, F1, PR AUC** (area under the precision–recall curve from `precision_recall_curve` + trapezoidal `auc`; and a few extras) from the local Langfuse exports next to `app.py`.

**Scope:** rows are restricted to the **latest calendar day** of score timestamps (`timestamp.max().date()`), i.e. the same **last snapshot / one batch** logic as `streamlit_app/app.py` (see Overview → last snapshot).

For the extra metrics we need the per-trace confusion label, exported as `error_type` with `value` in `{true_positive,false_positive,true_negative,false_negative}`. If it's missing/blank, re-run:

```bash
python export_langfuse_csv.py
```

`accuracy` alone isn’t enough to derive F1/precision/recall without those labels.

**LaTeX:** run the export cell (after `metrics_by_model` is built) to write `metrics_export/` (or `streamlit_app/metrics_export/` when CSVs live under `streamlit_app/`): `metrics_wide.csv`, `metrics_long.csv`, `by_metric/<metric>.csv` and `.tsv`, `metrics_tabular.tex` (paste or `\input` the tabular), `metrics_macros.tex` (`\newcommand{\bench<Model><Metric>}{…}`), and `export_meta.json`.


In [1]:
from __future__ import annotations

import re
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    auc,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
)

alt.data_transformers.disable_max_rows()

# Order for faceted charts (bar rows + daily small multiples)
METRIC_ORDER = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "specificity",
    "balanced_accuracy",
    "pr_auc",
]

In [2]:
def _first_existing_path(*candidates: str) -> Path:
    for cand in candidates:
        p = Path(cand)
        if p.exists():
            return p
    raise FileNotFoundError(f"None of these paths exist: {candidates}")


# Prefer streamlit_app/ first: a stray empty `langfuse_scores.csv` in the repo root would otherwise
# win over the real export next to this notebook (cwd-dependent footgun).
SCORES_CSV = _first_existing_path(
    "streamlit_app/langfuse_scores.csv",
    "langfuse_scores.csv",
)
TRACES_CSV = _first_existing_path(
    "streamlit_app/langfuse_traces.csv",
    "langfuse_traces.csv",
)

scores_raw = pd.read_csv(SCORES_CSV, low_memory=False)
traces_raw = pd.read_csv(TRACES_CSV, low_memory=False)

if scores_raw.empty or traces_raw.empty:
    raise ValueError(f"Loaded empty CSV — check paths: {SCORES_CSV=} {TRACES_CSV=}")

scores_raw.shape, traces_raw.shape

((57309, 6), (22390, 8))

In [3]:
def _pick_col(df: pd.DataFrame, *candidates: str) -> str:
    for cand in candidates:
        if cand in df.columns:
            return cand
    raise KeyError(f"None of these columns exist: {candidates}")


score_name_col = _pick_col(scores_raw, "name", "score_name")
trace_id_col = _pick_col(scores_raw, "traceId", "trace_id")
score_ts_col = _pick_col(scores_raw, "timestamp", "createdAt", "created_at")

trace_pk_col = _pick_col(traces_raw, "id", "trace_id")
trace_ts_col = _pick_col(traces_raw, "timestamp", "createdAt", "created_at")
trace_model_col = _pick_col(traces_raw, "metadata.model", "model")
trace_run_col = "metadata.run_id" if "metadata.run_id" in traces_raw.columns else ("run_id" if "run_id" in traces_raw.columns else None)

scores = scores_raw.copy()
scores["timestamp"] = pd.to_datetime(scores[score_ts_col], utc=True, errors="coerce")
scores = scores.dropna(subset=["timestamp"])

traces = traces_raw.copy()
traces["timestamp"] = pd.to_datetime(traces[trace_ts_col], utc=True, errors="coerce")
traces = traces.dropna(subset=["timestamp"])

traces_small = pd.DataFrame(
    {
        "trace_id": traces[trace_pk_col].astype(str).str.strip(),
        "model": traces[trace_model_col].fillna("unknown"),
        "run_id": traces[trace_run_col].fillna("") if trace_run_col else "",
        "trace_timestamp": traces["timestamp"],
    }
).drop_duplicates(subset=["trace_id"])

scores_small = scores[[trace_id_col, score_name_col, "timestamp"]].copy()
scores_small = scores_small.rename(columns={trace_id_col: "trace_id", score_name_col: "score_name"})
scores_small["trace_id"] = scores_small["trace_id"].astype(str).str.strip()

# keep all potentially relevant value columns for reconstruction
for col in ("value", "stringValue", "string_value", "valueString", "value_string", "comment"):
    if col in scores.columns:
        scores_small[col] = scores[col]
    else:
        scores_small[col] = np.nan

# Avoid mixed float/str inference in `value` (accuracy vs error_type) confusing downstream code.
if "value" in scores_small.columns:
    scores_small["value"] = scores_small["value"].astype("string")

df = scores_small.merge(traces_small, on="trace_id", how="left")
df = df[df["model"].notna() & (df["model"] != "unknown")].copy()

# Latest score day only — same batch as streamlit app "last snapshot" (max timestamp → calendar day).
_last_ts = df["timestamp"].max()
if pd.isna(_last_ts):
    raise ValueError("No valid score timestamps after merge.")
_last_day = pd.Timestamp(_last_ts).date()
df = df[df["timestamp"].dt.date == _last_day].copy()
if df.empty:
    raise ValueError(f"No score rows on last day {_last_day} — check CSV or timezone.")
print(f"Using last score day only: {_last_day}  ({len(df):,} score rows)")

df[["score_name", "model"]].value_counts().head(10)

Using last score day only: 2026-04-01  (9,630 score rows)


score_name  model            
error_type  gpt-5.2              300
            gpt-5-mini           300
accuracy    gpt-5-mini           300
cost_usd    gpt-5.2              300
            gpt-5-mini           300
accuracy    gpt-5.2              300
cost_usd    claude-sonnet-4-6    299
            gpt-5                298
            claude-sonnet-4-5    298
error_type  gpt-5                298
Name: count, dtype: int64

In [4]:
_COMMENT_RE = re.compile(r"expected\s*=\s*(True|False)\s*,\s*predicted\s*=\s*(True|False)")


def _parse_expected_predicted_from_comment(comment: str) -> tuple[bool, bool] | None:
    if not isinstance(comment, str):
        return None
    m = _COMMENT_RE.search(comment)
    if not m:
        return None
    expected = m.group(1) == "True"
    predicted = m.group(2) == "True"
    return expected, predicted


def _confusion_label_from_expected_predicted(expected: bool, predicted: bool) -> str:
    if expected is True and predicted is True:
        return "true_positive"
    if expected is True and predicted is False:
        return "false_negative"
    if expected is False and predicted is False:
        return "true_negative"
    return "false_positive"


def _coalesce_score_value(row: pd.Series, cols: tuple[str, ...]) -> str | None:
    """Langfuse CSV stores categorical scores (e.g. error_type) in `value`, not only in string* columns."""
    for c in cols:
        if c not in row.index:
            continue
        v = row[c]
        if pd.isna(v):
            continue
        s = str(v).strip()
        if s:
            return s
    return None


def build_trace_level_table(df_scores_and_traces: pd.DataFrame) -> pd.DataFrame:
    acc = df_scores_and_traces[df_scores_and_traces["score_name"] == "accuracy"].copy()
    err = df_scores_and_traces[df_scores_and_traces["score_name"] == "error_type"].copy()

    if acc.empty:
        raise ValueError("No 'accuracy' scores found.")

    # 1 row per trace (accuracy)
    acc = acc.sort_values("timestamp").drop_duplicates(subset=["trace_id"], keep="last")
    acc["correct"] = pd.to_numeric(acc["value"], errors="coerce")
    acc["correct"] = acc["correct"].fillna(0.0).clip(0.0, 1.0).astype(int)

    trace_tbl = acc[["trace_id", "timestamp", "model", "run_id", "correct"]].copy()
    trace_tbl = trace_tbl.rename(columns={"timestamp": "score_timestamp"})

    if err.empty:
        trace_tbl["confusion_label"] = np.nan
        trace_tbl["y_true"] = np.nan
        trace_tbl["y_pred"] = np.nan
        return trace_tbl

    err = err.sort_values("timestamp").drop_duplicates(subset=["trace_id"], keep="last")

    def _label_row(r: pd.Series) -> str | None:
        label = _coalesce_score_value(
            r,
            ("value", "stringValue", "string_value", "valueString", "value_string"),
        )
        if label in {"true_positive", "false_positive", "true_negative", "false_negative"}:
            return label
        parsed = _parse_expected_predicted_from_comment(r.get("comment"))
        if parsed:
            return _confusion_label_from_expected_predicted(*parsed)
        return None

    err["confusion_label"] = err.apply(_label_row, axis=1)

    err_small = err[["trace_id", "confusion_label"]]
    trace_tbl = trace_tbl.merge(err_small, on="trace_id", how="left")

    # Reconstruct y_true / y_pred from confusion label
    mapping = {
        "true_positive": (1, 1),
        "false_positive": (0, 1),
        "true_negative": (0, 0),
        "false_negative": (1, 0),
    }
    mapped = trace_tbl["confusion_label"].map(mapping)
    trace_tbl["y_true"] = mapped.map(lambda t: t[0] if isinstance(t, tuple) else np.nan)
    trace_tbl["y_pred"] = mapped.map(lambda t: t[1] if isinstance(t, tuple) else np.nan)

    return trace_tbl


traces_eval = build_trace_level_table(df)
traces_eval.head()

,trace_id,score_timestamp,model,run_id,correct,confusion_label,y_true,y_pred
0,2f33beb9-89f2-42bf-952c-77d50cdbeabb,2026-04-01 04:54:30.969000+00:00,gpt-5.4,gha-23832575964,1,None,NaN,NaN
1,2a213e56-ba50-40f5-b3ec-4a9f4516a07f,2026-04-01 04:54:30.975000+00:00,gpt-5.4,gha-23832575964,1,None,NaN,NaN
2,9769f268-ae87-4ef5-86c9-1103d5b822d5,2026-04-01 04:54:30.976000+00:00,gpt-5.4,gha-23832575964,1,None,NaN,NaN
3,79095c7c-d2d2-4d16-b2f7-0ad0ab1f074b,2026-04-01 04:54:30.977000+00:00,gpt-5.4,gha-23832575964,1,None,NaN,NaN
4,5e6dddf6-b9e7-4a7c-9a44-532e1bd0745b,2026-04-01 04:54:30.977000+00:00,gpt-5.4,gha-23832575964,1,None,NaN,NaN


In [5]:
def safe_metric(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Exception:
        return np.nan


def pr_auc_under_pr_curve(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Area under the precision–recall curve: trapezoidal integral ∫ precision d(recall)."""
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    return float(auc(recall, precision))


def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true = y_true[mask].astype(int)
    y_pred = y_pred[mask].astype(int)

    if y_true.size == 0:
        return {
            "n": 0,
            "tp": 0,
            "fp": 0,
            "tn": 0,
            "fn": 0,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "specificity": np.nan,
            "balanced_accuracy": np.nan,
            "pr_auc": np.nan,
        }

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    balanced_accuracy = np.nan
    if not np.isnan(specificity) and not np.isnan(sensitivity):
        balanced_accuracy = 0.5 * (specificity + sensitivity)

    # PR AUC = ∫ precision d(recall) on the curve from precision_recall_curve (trapezoidal rule via sklearn.metrics.auc).
    # y_score is predicted positive rate (here hard 0/1); with real probabilities the curve is richer.
    pr_auc = safe_metric(pr_auc_under_pr_curve, y_true, y_pred) if len(np.unique(y_true)) > 1 else np.nan

    return {
        "n": int(y_true.size),
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "balanced_accuracy": balanced_accuracy,
        "pr_auc": pr_auc,
    }


can_reconstruct = not traces_eval[["y_true", "y_pred"]].isna().all().all()
if not can_reconstruct:
    print(
        "Couldn't reconstruct y_true/y_pred from the current export.\n"
        "Re-run `python export_langfuse_csv.py` and ensure `error_type` is present with `value` in {true_positive,false_positive,true_negative,false_negative}.\n"
        "Showing accuracy-only until then."
    )

if can_reconstruct:
    # Avoid groupby.apply(Series): pandas 2.x can mis-shape the result so reindex() fills NaN everywhere.
    _rows: list[dict] = []
    for model, g in traces_eval.groupby("model", dropna=False):
        m = compute_binary_metrics(g["y_true"].to_numpy(), g["y_pred"].to_numpy())
        _rows.append({"model": model, **m})
    metrics_by_model = pd.DataFrame(_rows)
    expected_cols = [
        "model",
        "n",
        "tp",
        "fp",
        "tn",
        "fn",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "specificity",
        "balanced_accuracy",
        "pr_auc",
    ]
    metrics_by_model = metrics_by_model.reindex(columns=expected_cols)
else:
    metrics_by_model = (
        traces_eval.groupby(["model"], dropna=False)
        .agg(n=("trace_id", "count"), accuracy=("correct", "mean"))
        .reset_index()
    )
    for c in ("tp", "fp", "tn", "fn", "precision", "recall", "f1", "specificity", "balanced_accuracy", "pr_auc"):
        metrics_by_model[c] = np.nan

sort_metric = "f1" if can_reconstruct else "accuracy"
metrics_by_model.sort_values(sort_metric, ascending=False).head(20)

Couldn't reconstruct y_true/y_pred from the current export.
Re-run `python export_langfuse_csv.py` and ensure `error_type` is present with `value` in {true_positive,false_positive,true_negative,false_negative}.
Showing accuracy-only until then.


,model,n,accuracy,tp,fp,tn,fn,precision,recall,f1,specificity,balanced_accuracy,pr_auc
9,gpt-5.2,300,0.970000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,gpt-5,298,0.969799,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,gpt-5.4,298,0.959732,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,claude-sonnet-4-6,266,0.958647,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,gpt-5.1,296,0.956081,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,gpt-5-mini,300,0.906667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,claude-sonnet-4-5,265,0.905660,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,gpt-5-nano,297,0.861953,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,claude-haiku-4-5,264,0.837121,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,gpt-4.1-mini,297,0.801347,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
from datetime import datetime, timezone
import json


def _metrics_export_dir() -> Path:
    if Path("langfuse_scores.csv").is_file():
        return Path("metrics_export")
    if Path("streamlit_app/langfuse_scores.csv").is_file():
        return Path("streamlit_app/metrics_export")
    return Path("metrics_export")


_export_root = _metrics_export_dir()
_export_root.mkdir(parents=True, exist_ok=True)
_by_metric = _export_root / "by_metric"
_by_metric.mkdir(parents=True, exist_ok=True)

_batch_day = pd.Timestamp(traces_eval["score_timestamp"].max()).date().isoformat()

# --- Wide / long CSV (import into LaTeX via csvsimple, pgfplots, pandas, etc.)
metrics_by_model.to_csv(_export_root / "metrics_wide.csv", index=False)

_long_cols = [c for c in METRIC_ORDER if c in metrics_by_model.columns]
if _long_cols:
    metrics_long_export = metrics_by_model.melt(
        id_vars=["model"],
        value_vars=_long_cols,
        var_name="metric",
        value_name="value",
    )
    metrics_long_export.to_csv(_export_root / "metrics_long.csv", index=False)

# --- One file per metric: model \\t value (TSV) + CSV
for m in _long_cols:
    sub = metrics_by_model[["model", m]].dropna().copy()
    sub.to_csv(_by_metric / f"{m}.csv", index=False)
    tsv_lines = [f"{r['model']}\t{r[m]:.12g}" for _, r in sub.iterrows()]
    (_by_metric / f"{m}.tsv").write_text("\n".join(tsv_lines) + "\n", encoding="utf-8")

# --- LaTeX: tabular only (wrap in your own table / longtable)
_tex_cols = [c for c in ["model", "n", "tp", "fp", "tn", "fn", *METRIC_ORDER] if c in metrics_by_model.columns]
_tex_df = metrics_by_model[_tex_cols].copy()
_latex_tabular = _tex_df.to_latex(index=False, escape=True, float_format="%.4f")
(_export_root / "metrics_tabular.tex").write_text(
    "% Auto-generated — last score day " + _batch_day + "\n"
    "% \\input{metrics_tabular.tex} inside your table environment\n"
    + _latex_tabular
    + "\n",
    encoding="utf-8",
)

# --- \\newcommand per (model, metric) for inline numbers: \\benchGPTFourOneNanofOne
_lines = ["% Auto-generated — " + _batch_day, "% Usage: \\bench<ModelAlnum><MetricAlnum>"]
for _, row in metrics_by_model.iterrows():
    mslug = "".join(c for c in str(row["model"]) if c.isalnum())
    for mc in _long_cols:
        v = row[mc]
        if pd.isna(v):
            continue
        mcslug = "".join(c for c in mc if c.isalnum())
        _lines.append(f"\\newcommand{{\\bench{mslug}{mcslug}}}{{{v:.6f}}}")
(_export_root / "metrics_macros.tex").write_text("\n".join(_lines) + "\n", encoding="utf-8")

_meta = {
    "last_score_day": _batch_day,
    "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    "n_models": int(len(metrics_by_model)),
    "paths": {
        "metrics_wide_csv": str(_export_root / "metrics_wide.csv"),
        "metrics_long_csv": str(_export_root / "metrics_long.csv"),
        "metrics_tabular_tex": str(_export_root / "metrics_tabular.tex"),
        "metrics_macros_tex": str(_export_root / "metrics_macros.tex"),
        "by_metric_dir": str(_by_metric),
    },
}
(_export_root / "export_meta.json").write_text(json.dumps(_meta, indent=2), encoding="utf-8")

print("LaTeX-friendly export:", _export_root.resolve())

LaTeX-friendly export: /Users/georgiiburdi/DataspellProjects/benchmark_eval/streamlit_app/metrics_export


In [7]:
metrics_long = metrics_by_model.melt(
    id_vars=["model", "n", "tp", "fp", "tn", "fn"],
    value_vars=METRIC_ORDER,
    var_name="metric",
    value_name="value",
)

bar = (
    alt.Chart(metrics_long)
    .mark_bar()
    .encode(
        x=alt.X("value:Q", title=None, scale=alt.Scale(domain=[0, 1])),
        y=alt.Y("model:N", title=None, sort="-x"),
        color=alt.Color("model:N", legend=None),
        row=alt.Row("metric:N", title=None, sort=METRIC_ORDER),
        tooltip=[
            alt.Tooltip("model:N"),
            alt.Tooltip("metric:N"),
            alt.Tooltip("value:Q", format=".4f"),
            alt.Tooltip("n:Q"),
            alt.Tooltip("tp:Q"),
            alt.Tooltip("fp:Q"),
            alt.Tooltip("tn:Q"),
            alt.Tooltip("fn:Q"),
        ],
    )
    .properties(width=800)
)

bar

alt.Chart(...)

In [8]:
# Single calendar day in scope → per-model metrics are the bar chart above (no separate time series).
traces_eval["date"] = traces_eval["score_timestamp"].dt.floor("D")
print(
    "Date span in traces_eval:",
    traces_eval["score_timestamp"].min(),
    "→",
    traces_eval["score_timestamp"].max(),
)

Date span in traces_eval: 2026-04-01 04:54:30.969000+00:00 → 2026-04-01 04:59:46.466000+00:00


In [9]:
def _accuracy_chart_dir() -> Path:
    if Path("langfuse_scores.csv").is_file():
        return Path("accuracy_charts")
    if Path("streamlit_app/langfuse_scores.csv").is_file():
        return Path("streamlit_app/accuracy_charts")
    return Path("accuracy_charts")


_chart_dir = _accuracy_chart_dir()
_chart_dir.mkdir(parents=True, exist_ok=True)
_hist = _chart_dir / "history"
_hist.mkdir(parents=True, exist_ok=True)
_stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%d_%H%M%SZ")

try:
    for fname, chart in (("metrics_by_model", bar),):
        chart.save(str(_chart_dir / f"{fname}.png"), scale_factor=2)
        chart.save(str(_hist / f"{fname}_{_stamp}.png"), scale_factor=2)
    print("Latest PNGs:", _chart_dir.resolve())
    print("History:", _hist.resolve(), f"({_stamp})")
except Exception as exc:
    print("PNG export failed:", exc)
    print("Install: pip install vl-convert-python")

Latest PNGs: /Users/georgiiburdi/DataspellProjects/benchmark_eval/streamlit_app/accuracy_charts
History: /Users/georgiiburdi/DataspellProjects/benchmark_eval/streamlit_app/accuracy_charts/history (20260404_093957Z)
